# Task #4 - Context-Aware Chatbot Using LangChain or RAG    

### `Objective`:    
Build a conversational chatbot that can remember context and retrieve external information 
during conversations.    

### `Dataset`:    
Custom corpus (e.g., Wikipedia pages, internal documents, or any knowledge base) 

### `Instructions`:    
● Use LangChain or Retrieval-Augmented Generation (RAG)    
● Implement context memory for conversational history    
● Retrieve answers from a vectorized document store    
● Deploy the chatbot with Streamlit    

--------------------------

### Problem Statement:
Standard chatbots either rely solely on an LLM's pretrained knowledge (which can be outdated or lack domain-specific detail) or have no memory of earlier turns in a conversation, making multi-turn interactions feel disconnected and repetitive.

### Goal:
Build a context-aware chatbot that retrieves relevant information from a custom knowledge base (Wikipedia pages on Machine Learning, Artificial Intelligence, and Neural Networks) using Retrieval-Augmented Generation, while maintaining conversational memory across turns, and deploy it as an interactive Streamlit app with support for switching between multiple LLM providers.

## Installing and importing dependencies

In [15]:
# !pip install langchain.memory -q
# !pip install langchain langchain-community langchain-huggingface langchain-core requests wikipedia --quiet
# !pip install langchain-chroma chromadb --quiet

ERROR: Could not find a version that satisfies the requirement langchain.memory (from versions: none)
ERROR: No matching distribution found for langchain.memory


In [ ]:
import langchain
import requests

## Loading Wikipedia pages

In [2]:
from langchain_core.documents import Document

HEADERS = {"User-Agent": "ZainRAGChatbot/1.0 (student project; contact: your_email@example.com)"}

def fetch_full_wikipedia_page(title):
    url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "format": "json",
        "titles": title,
        "prop": "extracts",
        "explaintext": True,
    }
    response = requests.get(url, params=params, headers=HEADERS)
    data = response.json()
    pages = data["query"]["pages"]
    page = next(iter(pages.values()))
    return page.get("extract", "")

topics = ["Machine learning", "Artificial intelligence", "Neural network"]
documents = []

for topic in topics:
    content = fetch_full_wikipedia_page(topic)
    if content:
        documents.append(Document(page_content=content, metadata={"title": topic}))
        print(f"Loaded: {topic} ({len(content)} characters)")
    else:
        print(f"Failed to load: {topic}")

print(f"\nTotal documents loaded: {len(documents)}")

Loaded: Machine learning (58709 characters)
Loaded: Artificial intelligence (85280 characters)
Loaded: Neural network (5331 characters)

Total documents loaded: 3


## Split documents into chunks

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)
chunks = text_splitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks")

Split into 497 chunks


## Create embeddings and store in ChromaDB

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db",  # saves to disk automatically
)
print("Vector store created and persisted to /chroma_db")

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector store created and persisted to /chroma_db


In [8]:
vectorstore = Chroma(persist_directory="./chroma_db", embedding_function=embeddings)

## Testing Retrieval

In [9]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

results = retriever.invoke("What is a neural network?")
for r in results:
    print(r.metadata["title"], "-", r.page_content[:150], "...\n")

Neural network - A neural network is a group of interconnected units called neurons that send signals to one another. Neurons can be either biological cells or mathema ...

Neural network - In machine learning, a neural network is an artificial mathematical model used to approximate nonlinear functions. While early artificial neural netwo ...

Artificial intelligence - An artificial neural network is based on a collection of nodes also known as artificial neurons, which loosely model the neurons in a biological brain ...



## Setting Up API keys

In [63]:
from google.colab import userdata

hf_key = userdata.get("HF")
gem_key = userdata.get("GEM")
groq_key = userdata.get("GROQ")


## Setting up the LLM (Hugging_Face)

In [57]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

llm_endpoint = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation",
    huggingfacehub_api_token=hf_key
)
llm = ChatHuggingFace(llm=llm_endpoint)

## Setting up with Gemini

In [53]:
# !pip install langchain-google-genai --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 1.6 MB/s eta 0:00:00


In [59]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm2 = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.3,api_key=gem_key)

## Setting up with Groq

In [62]:
!pip install langchain-groq --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 2.2 MB/s eta 0:00:00


In [71]:
from langchain_groq import ChatGroq

llm3 = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.3,api_key=groq_key)
# "llama-3.1-8b-instant"


## Adding conversational memory

In [45]:
# from langchain.memory import ConversationBufferMemory
from langchain_classic.memory import ConversationBufferMemory

memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    output_key="answer",
)

## Combining Retrieval + Memory + LLM

In [76]:
from langchain_classic.chains import ConversationalRetrievalChain

qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm3, #llm:HF, llm2:Gemini, llm3:Groq
    retriever=retriever,
    memory=memory,
    return_source_documents=True,
)

result = qa_chain.invoke({"question": "What is Bill Gates?"})
print(result["answer"])

Bill Gates is a personality who, along with others such as Stephen Hawking and Elon Musk, has expressed concerns about existential risk from AI. He is likely referring to William Henry Gates III, the American business magnate and philanthropist, best known for co-founding Microsoft.


In [77]:
result2 = qa_chain.invoke({"question": "What is his wife name?"})
print(result2["answer"])

I don't know Bill Gates' current marital status or the name of his current or former wife based on the provided context. However, I can tell you that Bill Gates was previously married to Melinda Gates, but they divorced in 2021.


# Final Results

- Built a full RAG pipeline: Wikipedia pages fetched via a direct API call, split into 500-character chunks with overlap, embedded using all-MiniLM-L6-v2, and stored in a persistent ChromaDB vector store
- Verified retrieval quality — queries like "What is a neural network?" correctly pulled relevant chunks from the indexed articles
- Confirmed conversational memory works correctly — a follow-up question using a pronoun ("What is his wife's name?" after asking about Bill Gates) was correctly resolved using chat history, without needing to restate the subject
- Deployed as a Streamlit app with live switching between three LLM providers (Groq, Gemini, and Hugging Face), each maintaining its own independent conversation memory
- Added source attribution in the UI, showing which Wikipedia articles each answer was grounded in